# CORTEX-0 — GPU pretraining (Colab / Kaggle)

Trains the larger presets that are too slow on CPU. The first cell refuses to continue without a CUDA GPU.

Repository: `https://github.com/Frankenstein-Labs/Cortex-0`


In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit('No CUDA GPU detected. Select a GPU runtime (Colab: Runtime > Change runtime type > T4/A100).')
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
!git clone https://github.com/Frankenstein-Labs/Cortex-0.git /content/Cortex-0
%cd /content/Cortex-0
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q -e '.[data,serve]'

## 1. Data
Downloads TinyStories (CDLA-Sharing-1.0) and trains the 8,192-token BPE tokenizer. Reduce `--max-train-stories` if disk is limited.


In [ ]:
!mkdir -p data/raw
!curl -sL https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/data/train-00000-of-00004-2d5a1467fff1081b.parquet -o data/raw/tinystories_train.parquet
!curl -sL https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/data/validation-00000-of-00001-869c898b519ad725.parquet -o data/raw/tinystories_valid.parquet
!python scripts/prepare_data.py --max-train-stories 400000 --max-val-stories 5000 --vocab-size 8192

## 2. Pretrain
`cortex-0-small` is ~26.7M parameters. 8,000 steps is a short run; the preset default is 20,000.


In [ ]:
!python scripts/train.py --config configs/cortex-0-small.json --data data/processed --run-name cortex-0-small-gpu --steps 8000

## 3. Evaluate and sample
Metrics are measured on the validation split. Do not report them as a general capability claim.


In [ ]:
!python -m cortex.cli --checkpoint checkpoints/cortex-0-small-gpu/best.safetensors eval --data data/processed --max-batches 40
!python scripts/generate.py --checkpoint checkpoints/cortex-0-small-gpu/best.safetensors --prompt 'Once upon a time' --samples 3 --max-new-tokens 150

## 4. Optional: publish to Hugging Face
The token is read from Colab secrets at runtime and never written to disk. Skip this cell if no token is configured.


In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, upload_folder
token = userdata.get('HF_TOKEN')
api = HfApi(token=token)
api.create_repo('Frankenstein-Labs/Cortex-0', repo_type='model', exist_ok=True)
upload_folder(path_or_fileobj='checkpoints/cortex-0-small-gpu', path_in_repo='cortex-0-small-gpu', repo_id='Frankenstein-Labs/Cortex-0', repo_type='model', token=token)
upload_folder(path_or_fileobj='data/processed/tokenizer.json', path_in_repo='tokenizer/tokenizer.json', repo_id='Frankenstein-Labs/Cortex-0', repo_type='model', token=token)